# Will this GitHub issue get fixed?

For pandas issues still open 7 days after being opened, predict whether they'll eventually be closed as completed, using only information from those first 7 days. See the README for the full write-up.

The two download steps (sections 2 and 4) only run if their data files don't exist yet, so running the notebook top to bottom doesn't re-download anything.

## 1. Setup

In [1]:
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()                                   # reads GITHUB_TOKEN from .env
token = os.getenv("GITHUB_TOKEN")
headers = {"Authorization": f"Bearer {token}"}
REPO = "pandas-dev/pandas"

print("Token loaded:", token is not None)

Token loaded: True


## 2. Collect issues (runs once)

Rules applied here: skip pull requests, keep only issues at least 6 months old, stop at January 2024. GitHub stops page-by-page listing around page 100, so collection stops there.

In [2]:
RAW_FILE = "issues_raw.json"

if os.path.exists(RAW_FILE):
    print(f"{RAW_FILE} already exists, skipping download.")
else:
    CUTOFF_NEW = datetime.now(timezone.utc) - timedelta(days=180)   # at least 6 months old
    CUTOFF_OLD = datetime(2024, 1, 1, tzinfo=timezone.utc)

    rows, page = [], 1
    while True:
        resp = requests.get(
            f"https://api.github.com/repos/{REPO}/issues",
            headers=headers,
            params={"state": "all", "sort": "created", "direction": "desc",
                    "per_page": 100, "page": page},
        )
        batch = resp.json()
        if not isinstance(batch, list):
            print("Stopped by GitHub:", batch.get("message"))
            break
        if not batch:
            break

        for issue in batch:
            if "pull_request" in issue:
                continue                                   # skip pull requests
            created = datetime.fromisoformat(issue["created_at"].replace("Z", "+00:00"))
            if created > CUTOFF_NEW:
                continue                                   # too new to know the outcome
            rows.append({
                "number": issue["number"],
                "title": issue["title"],
                "body": issue["body"] or "",
                "created_at": issue["created_at"],
                "closed_at": issue["closed_at"],
                "state": issue["state"],
                "state_reason": issue.get("state_reason"),
                "author": issue["user"]["login"],
                "author_association": issue["author_association"],
                "labels": [l["name"] for l in issue["labels"]],
                "comments": issue["comments"],
            })

        oldest = datetime.fromisoformat(batch[-1]["created_at"].replace("Z", "+00:00"))
        print(f"page {page}: {len(rows)} issues kept, oldest {oldest.date()}")
        if oldest < CUTOFF_OLD:
            break
        page += 1
        time.sleep(0.5)

    raw = pd.DataFrame(rows)
    raw = raw[pd.to_datetime(raw["created_at"]) >= CUTOFF_OLD]
    raw.to_json(RAW_FILE, orient="records", lines=True)
    print("Saved", len(raw), "issues")

issues_raw.json already exists, skipping download.


## 3. Apply the rules and create the label

- Keep only issues still open on day 7 (never closed, or closed after day 7).
- Label: **fixed = 1** if closed as *completed*, otherwise **0**.

In [3]:
df = pd.read_json(RAW_FILE, lines=True)
df["created_at"] = pd.to_datetime(df["created_at"])
df["closed_at"] = pd.to_datetime(df["closed_at"])
df["days_to_close"] = (df["closed_at"] - df["created_at"]).dt.days

still_open_day7 = df["closed_at"].isna() | (df["days_to_close"] > 7)
df = df[still_open_day7]

df["fixed"] = ((df["state"] == "closed") & (df["state_reason"] == "completed")).astype(int)

print("Issues left:", len(df))
print(df["fixed"].value_counts())

Issues left: 1212
fixed
1    874
0    338
Name: count, dtype: int64


## 4. First-week comments (runs once)

One API call per issue. Only comments from the first 7 days are kept, so later comments like "Fixed in PR #1234" never enter the data.

In [4]:
WEEK1_FILE = "issues_week1.json"

def first_week_comments(issue_number, created_at):
    resp = requests.get(
        f"https://api.github.com/repos/{REPO}/issues/{issue_number}/comments",
        headers=headers,
        params={"per_page": 100},
    )
    comments = resp.json()
    if not isinstance(comments, list):
        return []
    cutoff = created_at + pd.Timedelta(days=7)
    return [
        {"author": c["user"]["login"], "author_association": c["author_association"],
         "created_at": c["created_at"], "body": c["body"] or ""}
        for c in comments
        if pd.to_datetime(c["created_at"]) <= cutoff      # first 7 days only
    ]

if os.path.exists(WEEK1_FILE):
    print(f"{WEEK1_FILE} already exists, skipping download.")
else:
    from tqdm.auto import tqdm
    all_comments = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        all_comments.append(first_week_comments(row["number"], row["created_at"]))
        time.sleep(0.2)
    df["week1_comments"] = all_comments
    df.to_json(WEEK1_FILE, orient="records", lines=True, date_format="iso")
    print("Saved", len(df), "issues with first-week comments")

df = pd.read_json(WEEK1_FILE, lines=True)
df["created_at"] = pd.to_datetime(df["created_at"])
print("Loaded", len(df), "issues")

issues_week1.json already exists, skipping download.
Loaded 1212 issues


## 5. Features

Only information available on day 7. Not used, because of leakage: total comment count (includes future comments) and labels (shown as they are today).

In [5]:
# How detailed is the description?
df["body_length"]      = df["body"].str.len()
df["has_code"]         = df["body"].str.contains("```").astype(int)
df["has_traceback"]    = df["body"].str.contains("Traceback").astype(int)
df["mentions_version"] = df["body"].str.contains(r"\d+\.\d+\.\d+").astype(int)

# Who opened it?
df["author_is_member"] = (df["author_association"] == "MEMBER").astype(int)

# First-week activity
def member_reply_hours(row):
    times = [pd.to_datetime(c["created_at"]) for c in row["week1_comments"]
             if c["author_association"] == "MEMBER"]
    if not times:
        return 168                       # no reply in week 1 = slowest possible, not 0
    return (min(times) - row["created_at"]).total_seconds() / 3600

df["hours_to_member_reply"] = df.apply(member_reply_hours, axis=1)
df["member_replied_week1"]  = (df["hours_to_member_reply"] < 168).astype(int)
df["n_week1_comments"]      = df["week1_comments"].apply(len)

# What kind of issue? Title prefix, rare ones grouped as OTHER
df["title_type"] = df["title"].str.extract(r"^([A-Za-z]+):")[0].str.upper().fillna("NONE")
df["title_type"] = df["title_type"].where(df["title_type"].isin(["BUG", "ENH", "DOC", "NONE"]), "OTHER")

## 6. Explore: fix rate by feature

In [6]:
for col in ["title_type", "member_replied_week1", "author_is_member"]:
    print(df.groupby(col)["fixed"].agg(["count", "mean"]).round(2), "\n")

            count  mean
title_type             
BUG           644  0.74
DOC           113  0.94
ENH           222  0.56
NONE           89  0.70
OTHER         144  0.72 

                      count  mean
member_replied_week1             
0                       480  0.73
1                       732  0.72 

                  count  mean
author_is_member             
0                   977  0.74
1                   235  0.64 



## 7. Split and baseline

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score

feature_cols = ["body_length", "has_code", "has_traceback", "mentions_version",
                "author_is_member", "member_replied_week1", "hours_to_member_reply",
                "n_week1_comments"]
X = pd.get_dummies(df[feature_cols + ["title_type"]], columns=["title_type"], dtype=int)
y = df["fixed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
print("Train:", len(X_train), " Test:", len(X_test), " Fixed rate:", round(y.mean(), 2))

lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Lazy model (always 'fixed') accuracy:", round(accuracy_score(y_train, lazy.predict(X_train)), 2))

Train: 969  Test: 243  Fixed rate: 0.72
Lazy model (always 'fixed') accuracy: 0.72


## 8. Compare models with cross-validation

The test set stays hidden here. Each model is scored on 5 splits of the training data.

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision tree":       DecisionTreeClassifier(max_depth=3, random_state=0),
    "Random forest":       RandomForestClassifier(n_estimators=300, max_depth=6, random_state=0),
    "Boosting":            HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                          max_iter=200, random_state=0),
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    print(f"{name:22s} AUC per fold: {scores.round(2)}  mean={scores.mean():.2f}  spread={scores.std():.2f}")

Logistic regression    AUC per fold: [0.56 0.61 0.65 0.61 0.63]  mean=0.61  spread=0.03
Decision tree          AUC per fold: [0.57 0.59 0.58 0.61 0.59]  mean=0.59  spread=0.01
Random forest          AUC per fold: [0.55 0.58 0.66 0.6  0.6 ]  mean=0.60  spread=0.04
Boosting               AUC per fold: [0.5  0.61 0.65 0.58 0.54]  mean=0.58  spread=0.05


## 9. Choose the cutoff (training data only)

Risk = chance of NOT being fixed. Missing an at-risk issue hurts more than a wrong flag, so recall is favored, without flagging most issues. Chosen cutoff: **0.30**.

In [9]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
p_fixed = cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1]
risk = 1 - p_fixed
at_risk = (y_train == 0).astype(int)

for cutoff in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    flag = (risk >= cutoff).astype(int)
    print(f"cutoff {cutoff:.2f}: flagged {flag.sum():4d}  "
          f"precision {precision_score(at_risk, flag, zero_division=0):.2f}  "
          f"recall {recall_score(at_risk, flag):.2f}")

cutoff 0.25: flagged  516  precision 0.34  recall 0.64
cutoff 0.30: flagged  341  precision 0.37  recall 0.46
cutoff 0.35: flagged  262  precision 0.38  recall 0.37
cutoff 0.40: flagged  135  precision 0.34  recall 0.17
cutoff 0.45: flagged   31  precision 0.45  recall 0.05
cutoff 0.50: flagged   21  precision 0.57  recall 0.04


## 10. Final test (run once)

The only time the hidden test set is used. Cross-validation (AUC about 0.62) is the more realistic estimate; this test split scored higher.

In [10]:
from sklearn.metrics import confusion_matrix

CUTOFF = 0.30

final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_model.fit(X_train, y_train)

risk_test = 1 - final_model.predict_proba(X_test)[:, 1]
at_risk_test = (y_test == 0).astype(int)
flag_test = (risk_test >= CUTOFF).astype(int)

print("Test AUC:  ", round(roc_auc_score(at_risk_test, risk_test), 2))
print("Flagged:   ", flag_test.sum(), "of", len(flag_test))
print("Precision: ", round(precision_score(at_risk_test, flag_test), 2))
print("Recall:    ", round(recall_score(at_risk_test, flag_test), 2))
print(confusion_matrix(at_risk_test, flag_test))

Test AUC:   0.67
Flagged:    91 of 243
Precision:  0.44
Recall:     0.59
[[124  51]
 [ 28  40]]
